# Synthetic Data Generator
### Step 11a : Build Synthetic Final Aligned Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_11b_build_final_aligned_observations_stage_code_reference.md`


In [1]:
# -----------------------------------------------------------------------------
# Stage 11B — Compact Synthetic Final Output
# -----------------------------------------------------------------------------
# Purpose:
# Build a Kaggle-like compact final output table.
#
# Source:
# - synthetic_sensor_observations_rebuilt_stage
#
# Target:
# - synthetic_sensor_observations_final_output

In [2]:
import os
import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
    execute_sql,
)

import pandas as pd

from utils.database.postgres import execute_sql, read_sql_dataframe

from utils.synthetic.pipeline.final_aligned_observation_writer import (
    build_final_aligned_observations_stage,
)

from utils.synthetic.pipeline.final_aligned_output_writer import build_synthetic_final_aligned_output_stage

from utils.synthetic.pipeline.final_aligned_incremental import run_final_align_loop

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

pd.set_option("display.max_colwidth", None)



In [3]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

IF_EXISTS_FLAG = "replace"

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_FINAL_ALIGN_COMPLETE_ONLY",
    True,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

REBUILT_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

FINAL_OUTPUT_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_OUTPUT_TABLE",
    "synthetic_sensor_observations_final_output",
)

N_SENSORS = NUMBER_OF_SENSORS

TIMESTAMP_OUTPUT_COLUMN = "timestamp"

MACHINE_STATUS_OUTPUT_COLUMN = "machine_status"

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}

print("Final aligned config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("rebuilt table:", REBUILT_TABLE_NAME)
print("final output table:", FINAL_OUTPUT_TABLE_NAME)
print("batch size:", OBSERVATION_WINDOW_SIZE)

Final aligned config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
rebuilt table: synthetic_sensor_observations_rebuilt_stage
final output table: synthetic_sensor_observations_final_output
batch size: 2500


---

In [4]:
engine = get_engine_from_env()

---

In [5]:
stage_11b_final_output_columns_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": FINAL_OUTPUT_TABLE_NAME,
    },
)

display(stage_11b_final_output_columns_dataframe)

,ordinal_position,column_name,data_type


In [6]:
'''
execute_sql(
    engine,
    f"""
    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE
    """
)

print(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")
'''

'\nexecute_sql(\n    engine,\n    f"""\n    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE\n    """\n)\n\nprint(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")\n'

In [7]:
# -----------------------------------------------------------------------------
# Build compact Kaggle-like synthetic output
# -----------------------------------------------------------------------------

stage_11b_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=FINAL_OUTPUT_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=N_SENSORS,
    complete_only=True,
    if_exists="replace",
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_output_column=TIMESTAMP_OUTPUT_COLUMN,
    machine_status_output_column=MACHINE_STATUS_OUTPUT_COLUMN,
)

display(pd.DataFrame([stage_11b_result]))

,status,dataset_id,run_id,rebuilt_rows_available,rebuilt_rows_read,final_rows_written,windows_processed,target_table,timestamp_output_column,machine_status_output_column
0,built,pump_synthetic_v1,synthetic_run_001,225000,225000,225000,90,synthetic_sensor_observations_final_output,timestamp,machine_status


----

In [8]:
# -----------------------------------------------------------------------------
# Validate compact final output
# -----------------------------------------------------------------------------

stage_11b_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        COUNT(*) FILTER (WHERE "{TIMESTAMP_OUTPUT_COLUMN}" IS NULL) AS null_timestamp_count,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" IS NULL) AS null_machine_status_count,
        MIN("{TIMESTAMP_OUTPUT_COLUMN}") AS min_timestamp,
        MAX("{TIMESTAMP_OUTPUT_COLUMN}") AS max_timestamp,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" = 'RECOVERING') AS recovering_rows,
        (
            COUNT(*) = 225000
            AND COUNT(*) FILTER (WHERE "{TIMESTAMP_OUTPUT_COLUMN}" IS NULL) = 0
            AND COUNT(*) FILTER (WHERE "{MACHINE_STATUS_OUTPUT_COLUMN}" IS NULL) = 0
        ) AS ready_for_export
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_validation_dataframe)

,row_count,dataset_id_count,run_id_count,asset_id_count,null_timestamp_count,null_machine_status_count,min_timestamp,max_timestamp,normal_rows,broken_rows,recovering_rows,ready_for_export
0,225000,1,1,1,0,0,2026-04-16 00:00:00+00:00,2026-09-19 05:59:00+00:00,210443,7,14550,True


---


### Sample Output

This shows the final Kaggle-shaped output table layout.


In [9]:
stage_11b_sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        "{TIMESTAMP_OUTPUT_COLUMN}",
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        "{MACHINE_STATUS_OUTPUT_COLUMN}"
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY "{TIMESTAMP_OUTPUT_COLUMN}", dataset_id, run_id, asset_id
    LIMIT 10
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_sample_dataframe)

,dataset_id,run_id,asset_id,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,machine_status
0,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:00:00+00:00,2.344329,48.327756,53.575950,43.424633,617.854335,NORMAL
1,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:01:00+00:00,2.316564,49.037209,50.378763,47.447264,617.673971,NORMAL
2,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:02:00+00:00,2.449135,48.920071,52.446083,46.391672,617.791244,NORMAL
3,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:03:00+00:00,2.489110,49.304505,51.872589,42.288103,617.794481,NORMAL
4,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:04:00+00:00,2.364396,45.769864,51.927495,41.465983,617.993983,NORMAL
5,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:05:00+00:00,2.223095,47.166849,50.717235,40.841192,617.698302,NORMAL
6,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:06:00+00:00,2.466576,48.560256,52.010692,44.414286,618.282081,NORMAL
7,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:07:00+00:00,2.337132,43.087177,52.973015,44.378573,618.379612,NORMAL
8,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:08:00+00:00,2.355965,45.514778,55.131609,43.304947,618.339956,NORMAL
9,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:09:00+00:00,2.270712,46.130963,53.648135,42.564812,617.956074,NORMAL


---

In [10]:
stage_11b_status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        "{MACHINE_STATUS_OUTPUT_COLUMN}" AS machine_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
    ORDER BY "{MACHINE_STATUS_OUTPUT_COLUMN}"
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11b_status_distribution_dataframe)

,machine_status,row_count
0,BROKEN,7
1,NORMAL,210443
2,RECOVERING,14550


---

In [11]:
# -----------------------------------------------------------------------------
# Preview compact final output
# -----------------------------------------------------------------------------

preview_final_output_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{FINAL_OUTPUT_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY "{TIMESTAMP_OUTPUT_COLUMN}"
    LIMIT 10
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(preview_final_output_dataframe)

,dataset_id,run_id,asset_id,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,...,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:00:00+00:00,2.344329,48.327756,53.575950,43.424633,617.854335,62.387073,...,47.973746,74.331914,59.387182,44.227766,37.686628,40.089498,43.412712,188.450560,300.841897,NORMAL
1,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:01:00+00:00,2.316564,49.037209,50.378763,47.447264,617.673971,75.127279,...,47.920156,57.487579,57.001130,41.540010,44.544907,39.875294,54.131620,161.877550,258.012292,NORMAL
2,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:02:00+00:00,2.449135,48.920071,52.446083,46.391672,617.791244,73.002240,...,47.809339,40.676645,53.673041,36.143568,50.311864,104.842210,53.682364,172.949255,195.420843,NORMAL
3,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:03:00+00:00,2.489110,49.304505,51.872589,42.288103,617.794481,70.965733,...,47.730139,45.527672,61.113795,58.907475,49.766001,67.184056,48.442577,165.008513,246.014069,NORMAL
4,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:04:00+00:00,2.364396,45.769864,51.927495,41.465983,617.993983,73.506156,...,47.667972,67.276336,32.353893,58.948834,48.398489,62.913080,52.980572,167.782434,303.756778,NORMAL
5,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:05:00+00:00,2.223095,47.166849,50.717235,40.841192,617.698302,72.078695,...,47.583781,59.022129,36.572340,64.405825,56.996448,164.436876,35.856502,NaN,262.211429,NORMAL
6,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:06:00+00:00,2.466576,48.560256,52.010692,44.414286,618.282081,69.288090,...,47.579444,45.631185,48.291763,60.284742,50.094292,84.666913,75.118331,NaN,259.461148,NORMAL
7,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:07:00+00:00,2.337132,43.087177,52.973015,44.378573,618.379612,84.899207,...,47.556396,40.954900,48.091975,54.867264,48.945672,121.905663,64.500765,215.333845,159.565991,NORMAL
8,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:08:00+00:00,2.355965,45.514778,55.131609,43.304947,618.339956,71.465360,...,47.478401,51.462160,43.118744,96.477474,53.834110,136.735674,71.894636,247.365130,292.713467,NORMAL
9,pump_synthetic_v1,synthetic_run_001,pump_asset_001,2026-04-16 00:09:00+00:00,2.270712,46.130963,53.648135,42.564812,617.956074,62.087797,...,47.495159,52.887860,48.346952,52.556832,50.457269,77.243923,47.485859,188.407151,219.427239,NORMAL


---

In [12]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

This support variant helps validate final-aligned output behavior for the synthetic path.
